In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b",
            temperature = 0.2)

load_dotenv()

In [ ]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`. You MUST use this tool
    to conclude the interaction.
    """
    return {"answer": answer, "tools_used": tools_used}

In [ ]:
add

In [ ]:
add.args_schema.model_json_schema()

When invoking the tool, a JSON string output by the LLM will be parsed into JSON and then consumed as kwargs, similar to the below:

In [ ]:
import json

llm_output_string = "{\"x\": 5, \"y\": 2}"  # this is the output from the LLM
llm_output_dict = json.loads(llm_output_string)  # load as dictionary
llm_output_dict

This is then passed into the tool function as `kwargs` (keyword arguments) as indicated by the `**` operator - the `**` operator is used to unpack the dictionary into keyword arguments.

In [ ]:
exponentiate.func(**llm_output_dict)

In [ ]:
tokens = []
async for token in llm.astream("What is NLP in 2 snetences?"):
    tokens.append(token)
    print(token.content, end="|", flush=True)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant. When answering a user's question "
        "you should first use one of the tools provided. After using a "
        "tool the tool output will be provided back to you. You MUST "
        "then use the final_answer tool to provide a final answer to the user. "
        "DO NOT use the same tool more than once."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [ ]:
from langchain_core.runnables.base import RunnableSerializable
from langchain_core.messages import BaseMessage

tools = [add, subtract, multiply, exponentiate, final_answer]

# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

chat_history = []
agent_scratchpad = []

In [ ]:
agent

In [ ]:
agent.invoke({
    "input": "hello my name is youssef",
    "chat_history": chat_history,
    "agent_scratchpad": agent_scratchpad
    })

In [ ]:
agent_with_memory.invoke({
    "input": "what was my name again? also what is 2+2",
    "chat_history": chat_history,
    "agent_scratchpad": agent_scratchpad
    })

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


# Create message history for memory
message_history = InMemoryChatMessageHistory()

# Wrap the agent with message history
agent_with_memory = RunnableWithMessageHistory(
    agent,
    lambda session_id: message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [ ]:
agent_with_memory.invoke(
    {"input": "What is 5 plus 3?"},
    config={"configurable": {"session_id": "session_1"}}
)

In [ ]:
# The next call will remember the conversation
agent_with_memory.invoke(
    {"input": "Multiply that result by 2"},
    config={"configurable": {"session_id": "session_1"}}
)

Executing the tool code requires two steps:

1. Map the tool `name` to the tool function.

2. Execute the tool function with the generated `args`.

In [ ]:
tool_call = agent.invoke({"input": "What is 10 + 10", "chat_history": []}) # -> this is the reasoning and tools agent
tool_call

In [ ]:
# create tool name to function mapping
name2tool = {tool.name: tool.func for tool in tools}

tool_exec_content = name2tool[tool_call.tool_calls[0]["name"]](
    **tool_call.tool_calls[0]["args"]
)
tool_exec_content

In [ ]:
from langchain_core.messages import ToolMessage

tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_exec_content}",
    tool_call_id=tool_call.tool_calls[0]["id"]
)

out = agent.invoke({
    "input": "What is 10 + 10",
    "chat_history": [],
    "agent_scratchpad": [tool_call, tool_exec]
})
out

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

retriever_a = vecstore_a.as_retriever()
retriever_b = vecstore_b.as_retriever()

retrieval = RunnableParallel(
    {
        "context_a": retriever_a, "context_b": retriever_b, "question": RunnablePassthrough()
    }
)